In [14]:
import pandas as pd

# real data
real_data = pd.read_csv('../datasets/real_data_PvDeepSet_ready.csv')
real_data['source'] = 'real'
real_data_train = real_data[(real_data.sets == 'train') & (real_data.sample_id_paired.str.contains('DF'))].reset_index(drop=True)
real_data_test = real_data[(real_data.sets == 'test') & (real_data.sample_id_paired.str.contains('DF'))].reset_index(drop=True)
real_data_test_lc = real_data[(real_data.sets == 'test') & (real_data.sample_id_paired.str.contains('LC'))].reset_index(drop=True)



pvAmpSeq_simultion_ready = pd.read_csv('../datasets/pvAmpSeq_simulation_ready.csv')


In [2]:
from DeepSets import run_prediction

In [3]:
pvAmpSeq_simultion_ready['prior_C'] = 1/3
pvAmpSeq_simultion_ready['prior_L'] = 1/3
pvAmpSeq_simultion_ready['prior_I'] = 1/3

In [23]:
real_data_test_lc[['sample_id_paired',	'pair_id',	'patient_id',	'sample_id']].drop_duplicates().head()

,sample_id_paired,pair_id,patient_id,sample_id
0,Sero-040-LC_4,Sero-040_4,Sero-040,Sero-040-R04
7,Sero-040-LC_4,Sero-040_4,Sero-040,Sero-040-R08
36,Sero-182-LC_2,Sero-182_2,Sero-182,Sero-182-R12
43,Sero-182-LC_2,Sero-182_2,Sero-182,Sero-182-R01
71,Sero-194-LC_2,Sero-194_2,Sero-194,Sero-194-R09


In [17]:
temp = real_data_test[real_data_test.country == 'Ethiopia'].copy().reset_index(drop=True)

temp['pair_order'] = temp['sample_id_paired'].str.split('_').str[1]
# temp['sample_id_paired'] = temp['sample_id_paired'].str[:9] + 'P' + temp['sample_id_paired'].str[11:]
# temp = temp[data.columns]


In [66]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
temp.sample_id_paired.nunique()

48

prepatring for model

In [11]:
import pandas as pd
real_data = pd.read_csv('../datasets/pvAmpSeq_simulation_ready.csv')
real_data

,patient_id,sample_id,episode,marker,allele,frequency,country,MOIs
0,Sero-040,Sero-040-R01,1.0,Chr01,Chr01-1,0.531524,ethiopia,6
1,Sero-182,Sero-182-R12,3.0,Chr11,Chr11-2,0.149320,ethiopia,3
2,Sero-182,Sero-182-R12,3.0,Chr13,Chr13-1,0.729045,ethiopia,3
3,Sero-182,Sero-182-R12,3.0,Chr13,Chr13-2,0.129575,ethiopia,3
4,Sero-182,Sero-182-R12,3.0,Chr14,Chr14-1,0.931269,ethiopia,3
...,...,...,...,...,...,...,...,...
6734,AR-322,AR-322-140,2.0,Chr03,Chr03-1,0.526098,solomon_island,2
6735,AR-322,AR-322-140,2.0,Chr02,Chr02-2,0.148837,solomon_island,2
6736,AR-322,AR-322-140,2.0,Chr02,Chr02-6,0.042156,solomon_island,2
6737,AR-302,AR-302-0,2.0,Chr11,Chr11-2,0.232755,solomon_island,2


In [17]:
real_data.columns.str.lower()

Index(['patient_id', 'sample_id', 'episode', 'marker', 'allele', 'frequency',
       'country', 'mois', 'loci'],
      dtype='object')

In [9]:
real_data.drop('marker', axis=1, inplace=True)

In [12]:
real_data['loci'] = 3

In [13]:
if 'loci' in real_data.columns and not 'marker' in real_data.columns:
    print(True)

In [2]:
# preparing data sample dataset for GUI test

sample_data = real_data[real_data.country != 'ethiopia']

sampled_patients = sample_data['patient_id'].sample(20)
mapping_dict = {}
for i, p_id in enumerate(sampled_patients):
    mapping_dict[p_id] = "P0" + str(i+1) if i <9 else "P" + str(i+1)

sample_data = sample_data[sample_data.patient_id.isin(sampled_patients)]
sample_data['patient_id'] = sample_data['patient_id'].map(mapping_dict)
sample_data = sample_data.drop(['sample_id', 'frequency', 'MOIs'], axis=1)


country_map = {'peru':'C1', 'solomon_island':'C2'}
sample_data['country'] = sample_data['country'].map(country_map)
sample_data = sample_data.sort_values('patient_id').reset_index(drop=True)
sample_data.to_csv('../datasets/GUI_sample_dataset.csv', index=False)

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
df = result['results_table']
df = df[['pair_id', 'prob_C', 'prob_L','prob_I','predicted_class']].drop_duplicates().reset_index(drop=True)
df.head()

,pair_id,prob_C,prob_L,prob_I,predicted_class
0,AR-024_P1,3.471411e-07,0.304471,6.955286e-01,I
1,AR-024_P10,3.380386e-09,0.173901,8.260994e-01,I
2,AR-024_P2,9.909202e-04,0.999007,2.018461e-06,L
3,AR-024_P3,6.369301e-01,0.363069,4.884979e-07,C
4,AR-024_P4,4.486354e-09,0.172445,8.275549e-01,I


In [ ]:
def apply_class_order(p_class):
    if p_class == 'C':
        return 1
    elif p_class == 'L':
        return 2
    else: return 3

df['class_order'] = df['predicted_class'].apply(apply_class_order)

df = df.sort_values(['class_order']).reset_index(drop=True)

In [ ]:
import plotly.express as px
import pandas as pd

df = result['results_table']
df = df[['pair_id', 'prob_C', 'prob_L','prob_I','predicted_class']].drop_duplicates().reset_index(drop=True)
df_C = df[df.predicted_class == 'C'].sort_values('prob_C', ascending=False).reset_index(drop=True)
df_L = df[df.predicted_class == 'L'].sort_values('prob_L', ascending=False).reset_index(drop=True)
df_I = df[df.predicted_class == 'I'].sort_values('prob_I', ascending=False).reset_index(drop=True)

df = pd.concat([df_C, df_L, df_I], ignore_index=True)

df_long = df.melt(
    id_vars=['pair_id'], 
    value_vars=['prob_C', 'prob_L', 'prob_I'],
    var_name='Probability_Type', 
    value_name='Probability'
)

In [77]:
import plotly.express as px
import pandas as pd

# 1. Prepare the data
# Assuming 'df' is your processed dataframe from the previous step
counts_df = df['predicted_class'].value_counts().reset_index()
counts_df.columns = ['Class', 'Count']

# 2. Create the Donut Chart
fig = px.pie(
    counts_df, 
    values='Count', 
    names='Class', 
    hole=0.5, # This creates the "donut" effect
    color='Class',
    # Keeping colors consistent: Orange (C), Yellow (L), Teal (I)
    color_discrete_map={
        'C': '#ff4d00', 
        'L': '#e6b400', 
        'I': '#1fb3c4'
    },
    template="plotly_white"
)

# 3. Refine the layout to match your previous style
fig.update_layout(
    margin=dict(l=20, r=20, t=60, b=20),
    height=500,
    paper_bgcolor='white',
    # Horizontal legend at the top
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.05,
        xanchor="center",
        x=0.5,
        title=None
    )
)

# 4. Enhance the traces (labels and percentages)
fig.update_traces(
    textposition='inside', 
    textinfo='percent+label', # Shows the Class name and %
    marker=dict(line=dict(color='#FFFFFF', width=2)) # Adds a small white gap between segments
)

fig.show()


In [74]:
import plotly.express as px
import pandas as pd

df = result['results_table']
df = df[['pair_id', 'prob_C', 'prob_L','prob_I','predicted_class']].drop_duplicates().reset_index(drop=True)

# Grouped and sorted blocks
df_C = df[df.predicted_class == 'C'].sort_values('prob_C', ascending=False).reset_index(drop=True)
df_L = df[df.predicted_class == 'L'].sort_values('prob_L', ascending=False).reset_index(drop=True)
df_I = df[df.predicted_class == 'I'].sort_values('prob_I', ascending=False).reset_index(drop=True)

df_sorted = pd.concat([df_C, df_L, df_I], ignore_index=True)

df_long = df_sorted.melt(
    id_vars=['pair_id'], 
    value_vars=['prob_C', 'prob_L', 'prob_I'],
    var_name='Probability_Type', 
    value_name='Probability'
)

fig = px.bar(
    df_long, 
    x="pair_id", 
    y="Probability", 
    color="Probability_Type",
    color_discrete_sequence=['#ff4d00', '#e6b400', '#1fb3c4'],
    category_orders={"pair_id": df_sorted['pair_id'].tolist()},
    template="plotly_white" # Sets white background and clean gridlines
)

fig.update_layout(
    title=None,
    margin=dict(l=50, r=20, t=60, b=50), # Adjusted for axis labels
    height=500,
    paper_bgcolor='white',
    plot_bgcolor='white',
    
    # Horizontal legend at the top
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        title=None
    ),
    
    xaxis=dict(
        title="Samples",      # Restored the label
        showticklabels=False, # Keeps specific IDs hidden for cleanliness
        ticks="",
        linecolor='black',    # Adds a crisp base line
        showgrid=False
    ),
    
    yaxis=dict(
        title="Probability",
        range=[0, 1],
        tickformat=".1f",
        linecolor='black',
        gridcolor='lightgrey' # Subtle grid for probability reference
    ),
    
    bargap=0
)

fig.show()

In [ ]:
result.keys()

In [6]:
from DeepSets import run_prediction

result = run_prediction(real_data, mode='prediction')
result['results_table']

143


,pair_id,patient_id,true_episode,prob_C,prob_L,prob_I,predicted_class
0,AR-024_P1,AR-024,1.0,3.471411e-07,0.304471,0.695529,I
1,AR-024_P1,AR-024,2.0,3.471411e-07,0.304471,0.695529,I
2,AR-024_P10,AR-024,5.0,3.380386e-09,0.173901,0.826099,I
3,AR-024_P10,AR-024,4.0,3.380386e-09,0.173901,0.826099,I
4,AR-024_P2,AR-024,1.0,9.909202e-04,0.999007,0.000002,L
...,...,...,...,...,...,...,...
783,Sero-277_P1,Sero-277,1.0,1.636915e-06,0.992661,0.007338,L
784,Sero-281_P1,Sero-281,1.0,2.164313e-08,0.304484,0.695516,I
785,Sero-281_P1,Sero-281,2.0,2.164313e-08,0.304484,0.695516,I
786,Sero-288_P1,Sero-288,1.0,3.749500e-08,0.155791,0.844209,I


In [ ]:
preprocessed_data[['sample_id_paired', 'patient_id', 'true_episode']].drop_duplicates().reset_index(drop=True)

In [ ]:
from DeepSets import run_prediction

result = run_prediction(real_data, mode='prediction')

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from DeepSets import DataPreprocessor, build_deepsets_tensors


def run_Prediction(data, mode='prediction'):

    # perform basic preprocess
    preprocessed_tensor = (
        data
        .pipe(DataPreprocessor.removing_single_episode_patients)
        .pipe(DataPreprocessor.calculate_population_Allele_frequencies, population='country')
        .pipe(DataPreprocessor.calculate_MOIs)
        .pipe(DataPreprocessor.paired_episode_assigner)
        .pipe(DataPreprocessor.build_deepsets_tensors, mode=mode)
    )
    loaded_data = DataPreprocessor.load_tensor(preprocessed_tensor)
    return loaded_data

result = run_Prediction(real_data)

In [ ]:
import json
# reading allele dictionary
with open('allele_dict.json', 'r') as file:
    allele_to_id = json.load(file)

In [ ]:
import torch
from DeepSets import PvDeepSets
# 1. Define the device
device = torch.device("cpu")

# 2. Initialize the model with the same parameters used during training
# Make sure len(allele_to_id) matches the training setup!
n_alleles = len(allele_to_id) 
model = PvDeepSets(n_alleles=n_alleles)
print(n_alleles)
# 3. Load the weights (state_dict)
# Replace "pv_deepsets_weights.pth" with your actual filename
model.load_state_dict(torch.load("../src/DeepSets/pv_deepsets_weights20260402.pth", map_location=device))



In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

def get_model_results(model, dataloader, device, threshold=0.5):
    model.eval()
    all_predicted = []
    y_pred_filtered = []
    undetermined_count = 0

    with torch.no_grad():
        for X_alleles, allele_mask, marker_mask, priors, MOI in dataloader:
            # ... (your existing GPU transfer code) ...
            y_pred = model(X_alleles, allele_mask, marker_mask, priors, MOI)
            
            # Apply Softmax if your model doesn't already output probabilities
            # probs = torch.softmax(y_pred, dim=1) 
            probs = y_pred 

            # Get max probabilities and their indices
            max_probs, preds = torch.max(probs, dim=1)

            for i in range(len(max_probs)):
                if max_probs[i] >= threshold:
                    y_pred_filtered.append(preds[i].item())
                else:
                    undetermined_count += 1

            all_predicted.append(y_pred.cpu().numpy())

    print(f"Total undetermined samples removed: {undetermined_count}")
    return np.vstack(all_predicted), y_pred_filtered

predicted_probs, y_pred_hard = get_model_results(model, result, device)




In [ ]:
y_pred_hard

In [ ]:
# Step 1: removing patients with only one episode
# Step 2: make all possible pair (including making episode 1&2)
# step 3: Calculate MOIs
# step 4: Calculate Afreq